In [0]:
from minisom import MiniSom

import plotly.express as px

In [0]:
ambiente = 'project'

EXTREME_THRESHOLD = 0.9
PORCENTAJE_SAMPLE_DATA_MONTH = 1
EVERY_N_YEARS = 10
RANDOM_SEED = 0
TEST_SIZE = 0.25

SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wave_energy', 'wave_power_kW_m']


In [0]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [0]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA_MONTH for row in data.select('coast_year_month').distinct().collect()}

df = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
)

In [0]:
p90_energia = df["energia_ola"].quantile(0.90)
p90_potencia = df["potencia_ola"].quantile(0.90)

In [0]:
df["es_extremo"] = (
    (df["energia_ola"] >= p90_energia) &
    (df["potencia_ola"] >= p90_potencia)
)
df_no_extremo = df[~df["es_extremo"]].copy()
